In [ ]:
from airport_wiki_scraper.routes_scraper import *
from airport_wiki_scraper.airroute import AirRoute
from airport_wiki_scraper.scraper import *
import mwparserfromhell as mwp
import json
from dotenv import load_dotenv
from pathlib import Path
from dataclasses import asdict
import sys

load_dotenv()

# Create directories (if not present)

In [ ]:
Path("../data/structured").mkdir(parents=True, exist_ok=True)
Path("../data/wikitext").mkdir(parents=True, exist_ok=True)

# Find list of airports with airport destination list on Wikipedia

In [ ]:
titles = fetch_apdl_article_titles()
len(titles)

In [ ]:
with open("../data/airports.txt", "w") as f:
    for title in titles:
        f.write(title + "\n")

# Retrieve articles

In [ ]:
with open("../data/airports.txt", "r") as f:
    articles_to_fetch = f.readlines()
articles_to_fetch = [line.strip() for line in articles_to_fetch]
articles = queue_fetch_articles(articles_to_fetch)

In [ ]:
with open("../data/wikitext/airport_articles.jsonl", "w") as f:
    for article in articles:
        article_json = json.dumps(article)
        f.write(article_json + "\n")

# Extract airlines and destinations from articles

In [ ]:
from datetime import datetime
def dt_serial(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

with open("../data/wikitext/airport_articles.jsonl", "r") as articles, open("../data/structured/airroutes.jsonl", "w") as routes_file:
    for line in articles:
        article = json.loads(line)
        print("Reading airport " + article["title"])
        wikitext = article["content"]
        passenger, _ = get_apdl(mwp.parse(wikitext))
        if passenger == None:
            print("Passenger list not found")
            continue
        try:
            air_routes = extract_apdl(passenger, article["title"])
            for route in air_routes:
                routes_file.write(json.dumps(asdict(route), default=dt_serial) + "\n")
        except Exception as e:
            raise Exception()